In [1]:
# -*- coding: utf-8 -*-
"""
Created on 2022-12-27
Revised on 2026-03-31

@author:       Oscar Trevizo
@institution:  Harvard Extension School — Graduate Data Science Program (2023)
@context:      Independent project — applying course concepts to real-world data
@environment:  Python 3.14.3 | myenv | MacBook Air M5

NOAA API Vignette
=================

Description:
    Demonstrates how to pull weather data from the NOAA National Weather
    Service (NWS) REST API using the noaa-sdk Python wrapper. Retrieves
    historical observations and forecasts (12-hr and hourly) for a given
    ZIP code and organizes results into time-indexed pandas DataFrames.

    Sections:
        1. Historical observations  — last 14 days (temp, wind, humidity,
                                      pressure, visibility, description)
        2. 12-hr forecasts          — temperature, wind, short description
        3. Hourly forecasts         — same fields, hourly resolution

References:
    - NOAA NWS API:      https://www.weather.gov/documentation/services-web-api
    - noaa-sdk (PyPI):   https://pypi.org/project/noaa-sdk/
    - noaa-sdk (GitHub): https://github.com/paulokuong/noaa
    - NWS Geolocation:   https://www.weather.gov/media/documentation/docs/NWS_Geolocation.pdf

Revision History:
    2022-12-27  Original development
                - Pull observations and forecasts via noaa-sdk
                - Build time-indexed DataFrames for obs, 12-hr, hourly

    2026-03-31  Revision
                - Added module-level docstring (Cell [1])
                - Confirmed compatibility with Python 3.14.3 / myenv / M5
                - Noted duplicate 'pressure' key in obs_df dict (cosmetic bug)
"""

"\nCreated on 2022-12-27\nRevised on 2026-03-31\n\n@author:       Oscar Trevizo\n@institution:  Harvard Extension School — Graduate Data Science Program (2023)\n@context:      Independent project — applying course concepts to real-world data\n@environment:  Python 3.14.3 | myenv | MacBook Air M5\n\nNOAA API Vignette\n=================\n\nDescription:\n    Demonstrates how to pull weather data from the NOAA National Weather\n    Service (NWS) REST API using the noaa-sdk Python wrapper. Retrieves\n    historical observations and forecasts (12-hr and hourly) for a given\n    ZIP code and organizes results into time-indexed pandas DataFrames.\n\n    Sections:\n        1. Historical observations  — last 14 days (temp, wind, humidity,\n                                      pressure, visibility, description)\n        2. 12-hr forecasts          — temperature, wind, short description\n        3. Hourly forecasts         — same fields, hourly resolution\n\nReferences:\n    - NOAA NWS API:      

# Import libraries

In [2]:
from noaa_sdk import noaa
import datetime
import json
import pandas as pd

/Users/otrevizo/.venvs/myenv/lib/python3.14/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.2.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


# Init parameters

In [3]:
# parameters for retrieving NOAA weather data
zip_code = '60610'
country = 'US'
today = datetime.datetime.now()
past = today - datetime.timedelta(days=14)
start_date = past.strftime("%Y-%m-%dT00:00:00Z")
end_date = today.strftime("%Y-%m-%dT23:59:59Z")

# Instantiate object to connect to REST API

In [4]:
weather = noaa.NOAA()

# Pull the data

In [5]:
observations = weather.get_observations(zip_code, country, start_date, end_date)

# Accumulate data in lists

The data comes in a complex JSON. The following command pulls certain values only, for simplicity.

In [6]:
# Lists to become columns in a DataFrame
time = []
zc = []   # Optionally =, one could loop through several zip codes
wind_speed = []
temperature = []
humidity = []
wind_direction = []
pressure = []
visibility = []
description = []

observations = weather.get_observations(zip_code, country, start_date, end_date)

for obs in observations:
    time.append(obs["timestamp"])
    zc.append(zip_code)
    wind_speed.append(obs["windSpeed"]["value"])
    temperature.append(obs["temperature"]["value"])
    humidity.append(obs["relativeHumidity"]["value"])
    wind_direction.append(obs["windDirection"]["value"])
    pressure.append(obs["barometricPressure"]["value"])
    visibility.append(obs["visibility"]["value"])
    description.append(obs["textDescription"])
    

# Build time series DataFrame

In [7]:
# Build the DataFrame using dictionary
obs_df = pd.DataFrame({'time':time, 'zip_code':zc, 'temperature':temperature, 'wind_speed':wind_speed, 'humidity':humidity,
                   'wind_direction':wind_direction, 'pressure':pressure, 
                   'visibility':visibility, 'description':description})

obs_df.time = pd.to_datetime(obs_df['time'])
obs_df.set_index('time', inplace=True)


obs_df.head()

,zip_code,temperature,wind_speed,humidity,wind_direction,pressure,visibility,description
time,,,,,,,,
2026-04-01 11:35:00+00:00,60610,3.0,27.792,86.702947,20.0,102133.48,16093.44,Cloudy
2026-04-01 11:30:00+00:00,60610,3.0,27.792,86.702947,30.0,102133.48,16093.44,Cloudy
2026-04-01 11:25:00+00:00,60610,3.0,31.500,86.702947,30.0,102099.62,16093.44,Cloudy and Windy
2026-04-01 11:23:00+00:00,60610,3.3,29.520,82.472738,30.0,102100.00,16090.00,Cloudy
2026-04-01 11:20:00+00:00,60610,3.0,27.792,86.702947,20.0,102099.62,16093.44,Cloudy


# Forecasts

# Accumulate 12-hr data in lists

In [8]:
# Lists to become columns in a DataFrame
time = []
zc = []   # Optionally =, one could loop through several zip codes
wind_speed = []
temperature = []
wind_direction = []
description = []

forecasts = weather.get_forecasts(zip_code, country, hourly=False, type='forecast')

for fcst in forecasts:
    # print(fcst)
    time.append(fcst["startTime"])
    zc.append(zip_code)
    temperature.append(fcst['temperature'])
    wind_speed.append(fcst["windSpeed"])
    wind_direction.append(fcst["windDirection"])
    description.append(fcst["shortForecast"])    


# Build time series forecast DataFrame

In [9]:
# Build the DataFrame using dictionary
fcst_df = pd.DataFrame({'time':time, 'zip_code':zc, 'temperature':temperature, 'wind_speed':wind_speed, 
                   'wind_direction':wind_direction, 'description':description})

fcst_df.time = pd.to_datetime(fcst_df['time'])
fcst_df.set_index('time', inplace=True)


fcst_df.head()

,zip_code,temperature,wind_speed,wind_direction,description
time,,,,,
2026-04-01 06:00:00-05:00,60610,41,15 to 20 mph,NNE,Chance Rain Showers
2026-04-01 18:00:00-05:00,60610,40,15 mph,E,Showers And Thunderstorms
2026-04-02 06:00:00-05:00,60610,70,15 to 25 mph,SSE,Showers And Thunderstorms
2026-04-02 18:00:00-05:00,60610,49,10 to 25 mph,SW,Showers And Thunderstorms
2026-04-03 06:00:00-05:00,60610,54,5 to 10 mph,SE,Chance Rain Showers


# Accumulate hourly data in lists

In [10]:
# Lists to become columns in a DataFrame
time = []
zc = []   # Optionally =, one could loop through several zip codes
wind_speed = []
temperature = []
wind_direction = []
description = []

forecasts = weather.get_forecasts(zip_code, country, hourly=True, type='forecastHourly')

for fcst in forecasts:
    # print(fcst)
    time.append(fcst["startTime"])
    zc.append(zip_code)
    temperature.append(fcst['temperature'])
    wind_speed.append(fcst["windSpeed"])
    wind_direction.append(fcst["windDirection"])
    description.append(fcst["shortForecast"])    


# Build time series forecast DataFrame

In [11]:
# Build the DataFrame using dictionary
hrly_fcst_df = pd.DataFrame({'time':time, 'zip_code':zc, 'temperature':temperature, 'wind_speed':wind_speed, 
                   'wind_direction':wind_direction, 'description':description})

hrly_fcst_df.time = pd.to_datetime(hrly_fcst_df['time'])
hrly_fcst_df.set_index('time', inplace=True)


hrly_fcst_df.head()

,zip_code,temperature,wind_speed,wind_direction,description
time,,,,,
2026-04-01 06:00:00-05:00,60610,37,15 mph,NNE,Cloudy
2026-04-01 07:00:00-05:00,60610,37,15 mph,NNE,Cloudy
2026-04-01 08:00:00-05:00,60610,38,15 mph,NNE,Slight Chance Rain Showers
2026-04-01 09:00:00-05:00,60610,38,15 mph,NE,Chance Rain Showers
2026-04-01 10:00:00-05:00,60610,39,20 mph,NE,Slight Chance Rain Showers
